# Hotel Guest Thread Analysis
**Pipeline:** Load → Detect threads → Classify (PROBLEM / QUESTION / OTHER) → Admin response time → Top 20 question topics

In [1]:
# ── Cell 0: Load & split ────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import os, json
from tqdm.auto import tqdm

PATH = "/home/nikita/code/PlatoIsDead/notebook_prototype/sentiment_analysis/Сообщения Vertical (5).xlsx"

df = pd.read_excel(PATH)
df["date_add"] = pd.to_datetime(df["date_add"])
df["is_admin"] = df["from_admin"].fillna(0).astype(int)
df["message"] = df["message"].astype(str).str.strip()
df = df[df["message"].str.len() > 0].copy()

# Split into guest and admin — both needed later
guest = df[df["is_admin"] == 0].copy()
admin = df[df["is_admin"] == 1].copy()

print(f"Total rows: {len(df)}")
print(f"Guest messages: {len(guest)}")
print(f"Admin messages: {len(admin)}")
print(f"Unique bookings: {df['ID_booking'].nunique()}")
print(f"Unique hotels: {df['hotel_id'].nunique()}")

Total rows: 90007
Guest messages: 56415
Admin messages: 33592
Unique bookings: 8187
Unique hotels: 5


In [2]:
# ── Cell 1: Thread detection ─────────────────────────────────────────────────
# A new thread starts when the gap between consecutive messages (any sender)
# within a booking exceeds GAP_HOURS.
# Using full df (not just guest) so admin replies don't break thread continuity.

GAP_HOURS = 4  # tune this — 4h means same-day topic shift starts a new thread

df_sorted = df.sort_values(["ID_booking", "date_add"]).reset_index(drop=True)

df_sorted["prev_time"] = df_sorted.groupby("ID_booking")["date_add"].shift(1)
df_sorted["gap_hrs"] = (
    (df_sorted["date_add"] - df_sorted["prev_time"]).dt.total_seconds() / 3600
)

# First message of a booking always starts thread 1
df_sorted["new_thread"] = (
    df_sorted["prev_time"].isna() |
    (df_sorted["gap_hrs"] > GAP_HOURS)
)
df_sorted["thread_id"] = df_sorted.groupby("ID_booking")["new_thread"].cumsum()

print(f"Total threads detected: {df_sorted.groupby(['ID_booking','thread_id']).ngroups}")
print(f"Avg threads per booking: {df_sorted.groupby(['ID_booking','thread_id']).ngroups / df_sorted['ID_booking'].nunique():.1f}")

# Quick sanity: threads with 1 message vs multi-message
thread_sizes = df_sorted.groupby(["ID_booking","thread_id"]).size()
print(f"Single-message threads: {(thread_sizes == 1).sum()}")
print(f"Multi-message threads:  {(thread_sizes > 1).sum()}")

Total threads detected: 22847
Avg threads per booking: 2.8
Single-message threads: 3890
Multi-message threads:  18957


In [ ]:
# ── Cell 2: Build threads DataFrame ─────────────────────────────────────────
# One row per (ID_booking, thread_id)
# text = all GUEST messages in thread joined — that's what we classify
# We keep metadata for later joins

guest_sorted = df_sorted[df_sorted["is_admin"] == 0].copy()

threads = (
    guest_sorted
    .groupby(["hotel_id", "ID_booking", "thread_id"])
    .agg(
        text=("message", lambda msgs: "\n---\n".join(msgs)),  # all guest msgs in thread
        n_guest_msgs=("message", "count"),
        thread_start=("date_add", "min"),
        thread_end=("date_add", "max"),
    )
    .reset_index()
)
=
print(f"Threads built: {len(threads)}")
threads.head(3)

Threads built: 21707


,hotel_id,ID_booking,thread_id,text,n_guest_msgs,thread_start,thread_end
0,1,29204,1,Тест\n---\nАоаоаоа\n---\nТестирую тест\n---\nЧ...,9,2023-10-05 14:23:16,2023-10-05 14:32:32
1,1,29204,2,Тест,1,2023-11-10 16:18:15,2023-11-10 16:18:15
2,1,29204,3,Тестовое сообщение\n---\nТестовое сообщение\n-...,18,2023-11-13 15:47:25,2023-11-13 17:18:58


In [4]:
# ── Cell 3: OpenAI thread classifier ────────────────────────────────────────
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

SYSTEM = """
You classify hotel guest conversation threads. The messages are in Russian.

PROBLEM: The guest needs physical action — someone must come or do something.
  Examples: cleaning request, broken AC, wifi voucher delivery, maintenance,
  key issue, noise from neighbors, "can I book a cleaner", "something is broken".
  Rule: if resolving it requires a person to act physically, it is PROBLEM.

QUESTION: A text reply fully resolves the issue — no physical action needed.
  Examples: checkout time, hotel policy, directions, parking, restaurant hours,
  wifi password (just the password, not a voucher), "where is the gym".

OTHER: Does not fit either category. You MUST explain why in 'reason'.
  Examples: pure greeting, payment dispute, cancellation, review threat,
  complaint with no actionable request.

Return ONLY valid JSON, no markdown:
{"category": "PROBLEM" | "QUESTION" | "OTHER", "reason": "brief explanation", "confidence": 0.0-1.0}
""".strip()


def classify_thread(text: str) -> dict:
    """Send one thread's guest messages to OpenAI, return classification dict."""
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            temperature=0,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": SYSTEM},
                {"role": "user", "content": text},
            ],
        )
        return json.loads(resp.choices[0].message.content)
    except Exception as e:
        return {"category": "ERROR", "reason": str(e), "confidence": 0.0}


print("classify_thread() ready")
print(f"Using model: {MODEL}")

classify_thread() ready
Using model: gpt-4o-mini


In [5]:
# ── Cell 4: Run classification on all threads ────────────────────────────────
# Checkpoint every 200 threads so you can resume if interrupted

CHECKPOINT_FILE = "threads_partial.parquet"
CHECKPOINT_EVERY = 200

categories, reasons, confidences = [], [], []

for i, row in tqdm(threads.iterrows(), total=len(threads)):
    result = classify_thread(row["text"])
    categories.append(result.get("category", "ERROR"))
    reasons.append(result.get("reason", ""))
    confidences.append(result.get("confidence", 0.0))

    # Checkpoint
    if (i + 1) % CHECKPOINT_EVERY == 0:
        threads_so_far = threads.iloc[:i+1].copy()
        threads_so_far["category"] = categories
        threads_so_far["reason"] = reasons
        threads_so_far["confidence"] = confidences
        threads_so_far.to_parquet(CHECKPOINT_FILE, index=False)
        print(f"  Checkpoint saved at row {i+1}")

threads["category"] = categories
threads["reason"] = reasons
threads["confidence"] = confidences

# Save final
threads.to_parquet("threads_classified.parquet", index=False)

print("\n=== Classification Results ===")
print(threads["category"].value_counts())
print()
print((threads["category"].value_counts(normalize=True) * 100).round(1))

  0%|          | 0/21707 [00:00<?, ?it/s]

  Checkpoint saved at row 200
  Checkpoint saved at row 400
  Checkpoint saved at row 600
  Checkpoint saved at row 800
  Checkpoint saved at row 1000
  Checkpoint saved at row 1200
  Checkpoint saved at row 1400
  Checkpoint saved at row 1600
  Checkpoint saved at row 1800
  Checkpoint saved at row 2000
  Checkpoint saved at row 2200
  Checkpoint saved at row 2400
  Checkpoint saved at row 2600
  Checkpoint saved at row 2800
  Checkpoint saved at row 3000
  Checkpoint saved at row 3200
  Checkpoint saved at row 3400
  Checkpoint saved at row 3600
  Checkpoint saved at row 3800
  Checkpoint saved at row 4000
  Checkpoint saved at row 4200
  Checkpoint saved at row 4400
  Checkpoint saved at row 4600
  Checkpoint saved at row 4800
  Checkpoint saved at row 5000
  Checkpoint saved at row 5200
  Checkpoint saved at row 5400
  Checkpoint saved at row 5600
  Checkpoint saved at row 5800
  Checkpoint saved at row 6000
  Checkpoint saved at row 6200
  Checkpoint saved at row 6400
  Checkpoint

In [6]:
# ── Cell 5: Admin response time ──────────────────────────────────────────────
# Grounded in actual timestamps — no invented scoring

admin_sorted = df_sorted[df_sorted["is_admin"] == 1].copy()

# First guest message per booking
first_guest = (
    guest_sorted.groupby(["hotel_id", "ID_booking"])["date_add"]
    .min().reset_index()
    .rename(columns={"date_add": "first_guest_time"})
)

# First admin reply per booking
first_admin = (
    admin_sorted.groupby(["hotel_id", "ID_booking"])["date_add"]
    .min().reset_index()
    .rename(columns={"date_add": "first_admin_time"})
)

# Admin message count per booking
admin_counts = (
    admin_sorted.groupby(["hotel_id", "ID_booking"])
    .size().reset_index(name="n_admin_msgs")
)

# Guest message count per booking
guest_counts = (
    guest_sorted.groupby(["hotel_id", "ID_booking"])
    .size().reset_index(name="n_guest_msgs")
)

# Build booking-level response stats
booking_stats = (
    first_guest
    .merge(first_admin, on=["hotel_id", "ID_booking"], how="left")
    .merge(admin_counts, on=["hotel_id", "ID_booking"], how="left")
    .merge(guest_counts, on=["hotel_id", "ID_booking"], how="left")
)

booking_stats["n_admin_msgs"] = booking_stats["n_admin_msgs"].fillna(0).astype(int)
booking_stats["admin_responded"] = booking_stats["n_admin_msgs"] > 0

booking_stats["reply_time_min"] = (
    (booking_stats["first_admin_time"] - booking_stats["first_guest_time"])
    .dt.total_seconds() / 60
).round(1)
# Negative = admin wrote first (edge case), set to NaN
booking_stats.loc[booking_stats["reply_time_min"] < 0, "reply_time_min"] = np.nan

print("=== Admin Response Stats (all bookings) ===")
total = len(booking_stats)
responded = booking_stats["admin_responded"].sum()
print(f"Bookings with admin reply: {responded} / {total} ({responded/total*100:.1f}%)")
print(f"Mean reply time:   {booking_stats['reply_time_min'].mean():.0f} min")
print(f"Median reply time: {booking_stats['reply_time_min'].median():.0f} min")

print("\n=== Per Hotel ===")
hotel_stats = booking_stats.groupby("hotel_id").agg(
    total_bookings=("ID_booking", "count"),
    response_rate=("admin_responded", "mean"),
    mean_reply_min=("reply_time_min", "mean"),
    median_reply_min=("reply_time_min", "median"),
    n_guest_msgs=("n_guest_msgs", "sum"),
    n_admin_msgs=("n_admin_msgs", "sum"),
).reset_index()
hotel_stats["response_rate"] = (hotel_stats["response_rate"] * 100).round(1)
hotel_stats = hotel_stats.round(1)
print(hotel_stats.to_string(index=False))

=== Admin Response Stats (all bookings) ===
Bookings with admin reply: 6695 / 7982 (83.9%)
Mean reply time:   878 min
Median reply time: 8 min

=== Per Hotel ===
 hotel_id  total_bookings  response_rate  mean_reply_min  median_reply_min  n_guest_msgs  n_admin_msgs
        1            2488           81.9          1201.9              19.6         12205          6917
        2            1009           80.2           186.1               5.4          3056          1878
        3             813           69.6          1604.2              15.3          2953          2005
        4             227           69.2           391.0              21.4           593           298
        5            3445           90.7           734.8               4.9         37608         22247


In [12]:
# Загружаем готовый результат
threads = pd.read_parquet("threads_classified.parquet")

# Берём только проблемы
problems = threads[threads["category"] == "PROBLEM"].copy()

# Топ-10 reasons по каждому отелю
# reason — это уже человеческое описание типа:
# "шум от соседей требует вмешательства персонала"
# "запрос на уборку номера"
# "сломан кондиционер"

top_problems = (
    problems
    .groupby(["hotel_id", "reason"])
    .size()
    .reset_index(name="n")
    .sort_values(["hotel_id", "n"], ascending=[True, False])
    .groupby("hotel_id")
    .head(10)
)
top_problems

,hotel_id,reason,n
757,1,"Гость запрашивает уборку номера, что требует ф...",50
764,1,"Гость запрашивает уборку, что требует физическ...",44
2082,1,"Гость сообщает о шуме от соседей, что требует ...",29
2084,1,"Гость сообщает о шуме от соседей, что требует ...",28
715,1,"Гость запрашивает уборку в номере, что требует...",25
2043,1,Гость сообщает о шуме от соседей и просит вмеш...,25
1619,1,"Гость сообщает о проблеме с горячей водой, что...",14
925,1,Гость запрашивает физическое действие — достав...,11
575,1,"Гость жалуется на шум от соседей, что требует ...",10
1085,1,Гость просит вмешательства для решения проблем...,8


In [7]:
# ── Cell 6: Top 20 question topics (TF-IDF + KMeans) ────────────────────────
# No LLM cost — pure statistics on the QUESTION threads
# Le Wagon NLP module: TF-IDF vectorization + clustering

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

question_texts = threads[threads["category"] == "QUESTION"]["text"].tolist()
print(f"QUESTION threads to cluster: {len(question_texts)}")

N_TOPICS = 20

vec = TfidfVectorizer(
    max_features=1000,
    ngram_range=(1, 2),       # unigrams + bigrams
    min_df=2,                  # ignore terms appearing in only 1 thread
    sublinear_tf=True,         # log scaling — helps with Russian chat
)
X = vec.fit_transform(question_texts)

km = KMeans(n_clusters=N_TOPICS, random_state=42, n_init=10)
km.fit(X)

terms = vec.get_feature_names_out()

print(f"\n=== Top {N_TOPICS} Question Topics ===")
for i, center in enumerate(km.cluster_centers_):
    top_indices = center.argsort()[-7:][::-1]  # top 7 terms per cluster
    top_terms = [terms[j] for j in top_indices]
    cluster_size = (km.labels_ == i).sum()
    print(f"Topic {i+1:2d} ({cluster_size:3d} threads): {', '.join(top_terms)}")

QUESTION threads to cluster: 4826

=== Top 20 Question Topics ===
Topic  1 (141 threads): до, скольки, до скольки, какого, до какого, какого числа, числа
Topic  2 (219 threads): есть, есть ли, ли, вас, пожалуйста есть, здравствуйте есть, ли вас
Topic  3 ( 49 threads): какое, на какое, какое число, число, уборка, какое время, запланирована
Topic  4 (257 threads): ваучера, код, код ваучера, можно код, пожалуйста, ваучера для, ваучера пожалуйста
Topic  5 (297 threads): вечер, добрый вечер, добрый, вечер подскажите, подскажите, спасибо, пожалуйста
Topic  6 (1042 threads): здравствуйте, на, можно, номер, спасибо, не, или
Topic  7 (196 threads): доброе, доброе утро, утро, утро подскажите, подскажите, спасибо, пожалуйста
Topic  8 (152 threads): пароль, пароль от, от, вайфая, от вайфая, какой, какой пароль
Topic  9 (172 threads): wi fi, wi, fi, как, подключиться, пароль, подключиться wi
Topic 10 (406 threads): день, добрый день, добрый, день подскажите, подскажите, подскажите пожалуйста, пожал

In [8]:
# ── Cell 7: OTHER threads — inspect reasons ──────────────────────────────────
# See what didn't fit PROBLEM or QUESTION — spot patterns manually

other = threads[threads["category"] == "OTHER"][["ID_booking", "thread_id", "text", "reason", "confidence"]]
print(f"OTHER threads: {len(other)}")
print()

# Print sample of reasons
for _, row in other.head(20).iterrows():
    print(f"[{row['ID_booking']} / thread {row['thread_id']}] {row['reason']}")
    print(f"  Text preview: {row['text'][:100].replace(chr(10), ' ')}")
    print()

OTHER threads: 5713

[29204 / thread 1] The messages are informal and do not contain any actionable requests or issues related to hotel services.
  Text preview: Тест --- Аоаоаоа --- Тестирую тест --- Чат не читать --- Ксения привет --- Как твои дела --- Епрст -

[29204 / thread 2] The message is a test and does not contain a request or issue related to hotel services.
  Text preview: Тест

[29204 / thread 3] The messages are test messages without any actionable request or specific issue.
  Text preview: Тестовое сообщение --- Тестовое сообщение --- Тестовое сообщение --- Тестовое сообщение --- Тестовое

[29204 / thread 4] The message is a test and does not contain a request or issue related to hotel services.
  Text preview: Тест

[29204 / thread 5] The messages are greetings and inquiries without actionable requests.
  Text preview: Ребята, а скажите плиз, как Дели доставка привозит еду поднимается на этаж или нет? --- всем привет 

[29204 / thread 6] The message is a test and does n